# INFS4203/7203 Project: Credit Default Classification

**Track 1: Data-oriented project**

Development notebook: exploratory data analysis, preprocessing trials, model comparison,
and hyperparameter tuning. The final chosen pipeline is implemented separately in `main.py`.

Random seed fixed at **42** throughout for reproducibility.

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 150)

SEED = 42
np.random.seed(SEED)

## 1. Data Loading and Initial Inspection

Load `development.csv` (labelled, used for all model selection/validation) and `test.csv`
(unlabelled, predictions only). Per the project spec, `test.csv` must not inform any
preprocessing or modelling decision — we only load it here to confirm its shape/columns
match expectations.

In [2]:
df = pd.read_csv('development.csv')
df_test = pd.read_csv('test.csv')

print('development.csv shape:', df.shape)
print('test.csv shape:', df_test.shape)

development.csv shape: (24000, 25)
test.csv shape: (6000, 24)


In [3]:
df.head(10)

,client_id,credit_limit,sex,education,marital_status,age,repayment_status_mar,repayment_status_feb,repayment_status_jan,repayment_status_dec,repayment_status_nov,repayment_status_oct,bill_amount_mar,bill_amount_feb,bill_amount_jan,bill_amount_dec,bill_amount_nov,bill_amount_oct,payment_amount_mar,payment_amount_feb,payment_amount_jan,payment_amount_dec,payment_amount_nov,payment_amount_oct,defaulted_next_month
0,22789,160000.0,female,NaN,single,33.0,payment_delayed_2_months,NaN,NaN,NaN,NaN,revolving_credit,161771.0,NaN,NaN,NaN,162681.0,163005.0,NaN,0.0,0.0,6100.0,12300.0,6100.0,no
1,29007,NaN,female,graduate_school,single,34.0,payment_delayed_1_month,NaN,paid_on_time,NaN,NaN,no_balance,0.0,NaN,0.0,0.0,0.0,0.0,53.0,0.0,0.0,0.0,0.0,0.0,no
2,16951,NaN,NaN,NaN,NaN,50.0,NaN,payment_delayed_2_months,revolving_credit,NaN,NaN,revolving_credit,10171.0,8150.0,NaN,8067.0,NaN,8400.0,2.0,1281.0,NaN,294.0,305.0,1000.0,no
3,22281,NaN,female,graduate_school,single,NaN,NaN,NaN,NaN,revolving_credit,NaN,NaN,206879.0,211037.0,215139.0,218513.0,131660.0,NaN,NaN,NaN,NaN,4800.0,NaN,6000.0,yes
4,11347,NaN,female,NaN,NaN,32.0,payment_delayed_1_month,no_balance,paid_on_time,revolving_credit,revolving_credit,no_balance,0.0,0.0,326.0,NaN,-235.0,-235.0,0.0,326.0,NaN,0.0,NaN,NaN,no
5,4103,NaN,NaN,NaN,single,NaN,paid_on_time,NaN,NaN,NaN,paid_on_time,payment_delayed_2_months,NaN,NaN,6649.0,5149.0,NaN,NaN,6959.0,15.0,0.0,NaN,0.0,NaN,yes
6,28655,NaN,NaN,high_school,NaN,33.0,paid_on_time,payment_delayed_2_months,NaN,payment_delayed_2_months,NaN,no_balance,NaN,755.0,44255.0,NaN,-772.0,-772.0,NaN,43500.0,0.0,0.0,NaN,NaN,no
7,26151,50000.0,NaN,university,NaN,NaN,payment_delayed_3_months,payment_delayed_3_months,payment_delayed_2_months,NaN,NaN,payment_delayed_2_months,47904.0,NaN,48602.0,NaN,44837.0,44535.0,2000.0,NaN,NaN,2000.0,2000.0,1500.0,yes
8,15092,360000.0,male,graduate_school,NaN,NaN,revolving_credit,NaN,revolving_credit,revolving_credit,NaN,revolving_credit,120533.0,124530.0,128518.0,134368.0,135626.0,134864.0,NaN,6000.0,NaN,4918.0,NaN,3300.0,no
9,6267,320000.0,NaN,university,single,29.0,revolving_credit,NaN,revolving_credit,revolving_credit,revolving_credit,NaN,324693.0,NaN,325024.0,NaN,297637.0,301575.0,NaN,NaN,11900.0,10503.0,10886.0,10040.0,no


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24000 entries, 0 to 23999
Data columns (total 25 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   client_id             24000 non-null  int64  
 1   credit_limit          14400 non-null  float64
 2   sex                   14400 non-null  object 
 3   education             14400 non-null  object 
 4   marital_status        14400 non-null  object 
 5   age                   14400 non-null  float64
 6   repayment_status_mar  14400 non-null  object 
 7   repayment_status_feb  14400 non-null  object 
 8   repayment_status_jan  14400 non-null  object 
 9   repayment_status_dec  14400 non-null  object 
 10  repayment_status_nov  14400 non-null  object 
 11  repayment_status_oct  14400 non-null  object 
 12  bill_amount_mar       14400 non-null  float64
 13  bill_amount_feb       14400 non-null  float64
 14  bill_amount_jan       14400 non-null  float64
 15  bill_amount_dec    

In [5]:
df.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
client_id,24000.0,NaN,NaN,NaN,14973.683417,8641.664947,1.0,7513.75,14966.0,22438.25,30000.0
credit_limit,14400.0,NaN,NaN,NaN,166951.527778,129539.486324,10000.0,50000.0,140000.0,240000.0,1000000.0
sex,14400,2,female,8766,NaN,NaN,NaN,NaN,NaN,NaN,NaN
education,14400,5,university,6802,NaN,NaN,NaN,NaN,NaN,NaN,NaN
marital_status,14400,4,single,7699,NaN,NaN,NaN,NaN,NaN,NaN,NaN
age,14400.0,NaN,NaN,NaN,35.359097,9.179001,21.0,28.0,34.0,41.0,74.0
repayment_status_mar,14400,11,revolving_credit,7002,NaN,NaN,NaN,NaN,NaN,NaN,NaN
repayment_status_feb,14400,11,revolving_credit,7590,NaN,NaN,NaN,NaN,NaN,NaN,NaN
repayment_status_jan,14400,11,revolving_credit,7537,NaN,NaN,NaN,NaN,NaN,NaN,NaN
repayment_status_dec,14400,10,revolving_credit,7921,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### 1.1 Column roles

Split columns into their declared roles per the spec, so later cells (missingness,
distributions, preprocessing) can operate on the right groups. `client_id` is an identifier
only and must never be used as a predictor.

In [6]:
id_col = 'client_id'
target_col = 'defaulted_next_month'

numeric_cols = ['credit_limit', 'age'] + \
    [f'bill_amount_{m}' for m in ['oct', 'nov', 'dec', 'jan', 'feb', 'mar']] + \
    [f'payment_amount_{m}' for m in ['oct', 'nov', 'dec', 'jan', 'feb', 'mar']]

categorical_cols = ['sex', 'education', 'marital_status']

repayment_status_cols = [f'repayment_status_{m}' for m in ['oct', 'nov', 'dec', 'jan', 'feb', 'mar']]

predictor_cols = numeric_cols + categorical_cols + repayment_status_cols

assert set([id_col, target_col] + predictor_cols) == set(df.columns), 'Column split does not match development.csv columns'
assert set([id_col] + predictor_cols) == set(df_test.columns), 'Column split does not match test.csv columns'

print(f'{len(numeric_cols)} numeric, {len(categorical_cols)} categorical, '
      f'{len(repayment_status_cols)} repayment-status, 1 identifier, 1 target')

14 numeric, 3 categorical, 6 repayment-status, 1 identifier, 1 target


### 1.2 Missing values

Count and proportion of missing values per column, in `development.csv`. This informs the
imputation strategies to trial later (Section on Preprocessing). Note: any imputation must be
fit only within each CV training fold, this cell is descriptive/exploratory only.

In [ ]:
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
missing_summary = pd.DataFrame({'n_missing': missing, 'pct_missing': missing_pct})
missing_summary[missing_summary['n_missing'] > 0]

In [ ]:
rows_all_missing = df[predictor_cols].isnull().all(axis=1).sum()
rows_any_missing = df[predictor_cols].isnull().any(axis=1).sum()
print(f'Rows missing ALL predictors: {rows_all_missing} ({rows_all_missing / len(df) * 100:.1f}%)')
print(f'Rows missing AT LEAST ONE predictor: {rows_any_missing} ({rows_any_missing / len(df) * 100:.1f}%)')

**Observation:** every predictor column is independently missing in exactly 9,600 rows (40%),
but these are *not* the same rows across columns, only 17  of the rows have every predictor missing,
while 23,586 rows (98.3%) are missing at least one value somewhere. Row-wise deletion
(`dropna()`) is therefore not viable. It would discard almost the entire dataset. Imputation
could be a good option, to be trialled per-fold in the Preprocessing section (mean/median
vs. class-conditional, for numeric and mode vs. a dedicated "missing" category, for categorical).

### 1.3 Target class balance

`defaulted_next_month` is the target (`yes` = default, `no` = no default). Class imbalance
is expected which is we why we optimise for positive-class F1 rather than
accuracy, and why stratified CV is required.

In [ ]:
target_counts = df[target_col].value_counts()
target_pct = (df[target_col].value_counts(normalize=True) * 100).round(2)
pd.DataFrame({'count': target_counts, 'pct': target_pct})

### 1.4 Categorical / repayment-status value inspection

Look at the unique values for `sex`, `education`, `marital_status`, and the six
`repayment_status_*` columns. This is the first look needed to decide ordinal vs. one-hot
encoding for repayment status (see open decision in project notes).

In [ ]:
for col in categorical_cols + repayment_status_cols:
    print(f'--- {col} ---')
    print(df[col].value_counts(dropna=False))
    print()